# 29 · CRAG / Self-RAG / Adaptive RAG

> Agentic RAG 的三位重要成员：一个**纠偏**、一个**自省**、一个**路由**。

**本文件覆盖知识点**：CRAG(Corrective RAG) / Retrieval Grader / Web Search / Self-RAG / Retrieval Decision·Relevance·Support·Critique / Adaptive RAG

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. CRAG：先评测检索，不好就纠正

```text
Retrieve → Evaluate Retrieval(Retrieval Grader)
                 │
            好? ├─ Yes → Generate
                 └─ No  → Correct：改写查询重试 / 触发 Web Search 补外部知识
```

- **Retrieval Grader**：让 LLM 给“检索到的片段是否与问题相关”打分（0/1 或分级）；
- 相关度高 → 直接用；部分相关 → 合并；不相关 → **纠偏**（重写查询再试，或联网）。

In [ ]:
# CRAG 的“检索质量分诊”骨架
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def grade(query, doc):
    """Retrieval Grader: 返回 1(相关) / 0(不相关)"""
    from dashscope import Generation
    p = (f'问题: {query}\n文档: {doc}\n该文档是否与问题相关？只输出 1 或 0。')
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':p}],
                        api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content.strip()

def crag(query, docs):
    relevant = [d for d in docs if grade(query, d) == '1']
    if relevant:
        return 'generate', relevant
    return 'correct', []   # 触发: 改写查询 / web search

if API_KEY and '你的' not in API_KEY:
    action, rel = crag('私有化部署', ['星云支持公有云与私有化两种部署', '牛肉面的做法'])
    print('分诊结果:', action, '| 可用片段数:', len(rel))
else:
    print('CRAG 分诊：不相关文档会被剔除 → 全部不相关则触发纠偏/联网。')

## 2. Self-RAG：检索要不要？检索有没有用？

让 LLM 输出**反思标记**（token），自主决策：

```text
[检索决策] 这个问题需要外部知识吗?   → 需要/不需要
[Relevance] 检索到的片段相关吗?       → 相关/无关(丢弃)
[Support]   回答被片段支持吗?        → 有据/无据
[Critique]  整体够不够?              → 继续搜或停止
```

- 需要才检索 → 减少不必要的检索成本；
- 片段无关就丢 → 抑制噪声与幻觉。

In [ ]:
# 知识点·真调说明：Self-RAG 反思标记 —— 让模型自检“片段相关吗？回答被证据支持吗？”
import json as _json
_q = '星云支持私有化部署吗？'
_chunk = '星云机器人提供公有云与私有化两种部署方式；私有化部署需联系销售评估环境。'
_ans = '星云支持私有化部署，且私有化版本附带 7×24 专家驻场与免费硬件扩容。'
print('问题：', _q)
print('检索片段：', _chunk)
print('候选回答：', _ans)
print()
out = _llm_live(
    prompt='问题：' + _q + '\n检索片段：' + _chunk + '\n候选回答：' + _ans +
           '\n请按“反思标记”逐项自检，只输出一个 JSON 对象。',
    system='你是 Self-RAG 反思器。只依据“检索片段”判断候选回答是否有据，不要被候选回答本身说服。'
           '输出 JSON：{"need_retrieval": 布尔, "is_relevant": 布尔, "is_supported": 布尔, '
           '"verdict": "revise / continue / answer 等结论", '
           '"evidence": "指出候选回答中哪句没有被片段支持"}。只输出 JSON 对象，禁止其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"need_retrieval": true, "is_relevant": true, "is_supported": false, '
             '"verdict": "revise", '
             '"evidence": "片段只说提供私有化部署，并未提到 7×24 专家驻场与免费硬件扩容，这两句无据"}',
    temperature=0.2,
)
if out is None:
    out = ('{"need_retrieval": true, "is_relevant": true, "is_supported": false, '
           '"verdict": "revise", '
           '"evidence": "片段只说提供私有化部署，并未提到 7×24 专家驻场与免费硬件扩容，这两句无据"}')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _r = _json.loads(out)
    print('json.loads 通过 ✅ need_retrieval=%s | is_relevant=%s | is_supported=%s' % (
        _r.get('need_retrieval'), _r.get('is_relevant'), _r.get('is_supported')))
    print('verdict：', _r.get('verdict'))
    print('证据缺口：', _r.get('evidence'))
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 若直接让模型作答，它可能把“驻场/免费扩容”一并写进答案；Self-RAG 用 [Support]/[Critique] 这类反思标记'
      '拦下“无据”句子再修正——这正是它抑制幻觉的手段。')

## 3. Adaptive RAG：按问题类型路由

```text
问题 → 路由(Router)
   ├─ 简单/直答 → 普通 RAG
   ├─ 复杂多跳 → Query Decomposition / Agent
   ├─ 知识过时 → Web Search / CRAG
   └─ 关系型   → Graph RAG
```

路由判据可以是 **分类器 LLM**（“这是事实题还是综述题”）或规则（关键词/元数据）。



In [ ]:
# 知识点·真调说明：Adaptive RAG 路由 —— 真调模型把问题分诊到最合适的检索/问答方案
import json as _json
_qs = [
    '2025 年销售额最高的商品是哪个？（需查数据库）',
    '通义千问和文心一言现在哪个更强？（观点易过时，需最新资料）',
    '阿里云都收购了哪些公司？各自的 AI 产品线是什么？（跨文档多跳关系）',
    '星云机器人支持私有化部署吗？（查内部产品手册）',
]
print('待路由的问题：')
for _i, _qq in enumerate(_qs, 1):
    print('  %d) %s' % (_i, _qq))
print()
out = _llm_live(
    prompt='为下面 4 个问题各选一个最合适的检索/问答方案，只输出一个 JSON 数组：\n' +
           '\n'.join('%d) %s' % (i + 1, q) for i, q in enumerate(_qs)) +
           '\n可选方案含义：plain_rag=普通向量检索；agentic=多轮拆解后检索；web_search=联网取最新；'
           'graph_rag=图谱关系检索；sql=查数据库。',
    system='你是 Adaptive RAG 的路由器。输出 JSON 数组，每项 {"qid": 数字, "route": "方案名", "reason": "一句话"}。'
           '只依据问题性质选择，禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '[{"qid": 1, "route": "sql", "reason": "数值聚合，答案在数据库"}, '
             '{"qid": 2, "route": "web_search", "reason": "实时对比，静态文档会过时"}, '
             '{"qid": 3, "route": "graph_rag", "reason": "跨实体多跳关系"}, '
             '{"qid": 4, "route": "plain_rag", "reason": "产品手册类，普通向量检索即可"}]',
    temperature=0.1,
)
if out is None:
    out = ('[{"qid": 1, "route": "sql", "reason": "数值聚合，答案在数据库"}, '
           '{"qid": 2, "route": "web_search", "reason": "实时对比，静态文档会过时"}, '
           '{"qid": 3, "route": "graph_rag", "reason": "跨实体多跳关系"}, '
           '{"qid": 4, "route": "plain_rag", "reason": "产品手册类，普通向量检索即可"}]')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _rr = _json.loads(out)
    print('json.loads 通过 ✅ 路由结果：')
    for _x in sorted(_rr, key=lambda z: z['qid']):
        print('  问题%d -> %s（%s）' % (_x['qid'], _x['route'], _x['reason']))
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 同一入口按“问题类型”分流：该查库的查库、该联网的联网、该走图的多跳——这就是 Adaptive RAG 的路由器，'
      '也呼应 31 课“文档走 RAG / 数值走 SQL”的并用。')

## 小结

- **CRAG** 事后纠偏（含联网）；**Self-RAG** 事中自省；**Adaptive RAG** 事前路由；
- 三者可组合；共同点：**让模型参与“要不要检索/检索得怎么样”的决策**。